In [28]:
import sys
from pathlib import Path
import pandas as pd
from contextlib import contextmanager

projectRoot = Path().resolve().parents[0]
print(f"path: {projectRoot}")
sys.path.append(str(projectRoot))

@contextmanager
def full_df_display():
    with pd.option_context(
        "display.max_columns", None,
        "display.max_colwidth", None,
        "display.expand_frame_repr", False,
    ):
        yield

%load_ext autoreload
%autoreload 2

path: /workspace
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
from src.data_download.companies_house import search_company
result = search_company("08948140")
result

{'items': [{'kind': 'searchresults#company',
   'description_identifier': ['incorporated-on'],
   'company_status': 'active',
   'date_of_creation': '2014-03-19',
   'company_type': 'ltd',
   'company_number': '08948140',
   'address': {'address_line_1': 'Prime Park Way',
    'country': 'England',
    'locality': 'Derby',
    'postal_code': 'DE1 3QB',
    'premises': '17'},
   'title': 'FINANCE ADVICE CENTRE LTD',
   'address_snippet': '17 Prime Park Way, Derby, England, DE1 3QB',
   'description': '08948140 - Incorporated on 19 March 2014',
   'links': {'self': '/company/08948140'}}],
 'kind': 'search#companies',
 'page_number': 1,
 'items_per_page': 20,
 'total_results': 1,
 'start_index': 0}

In [30]:
from src.data_download.companies_house import download_full_filing_history

# Extract the first company name and number from the filtered dataframe
company_name, company_number = "FINANCE ADVICE CENTRE LTD", "08948140"

# Download the full filing history for the selected company
download_full_filing_history(company_name, company_number)

PosixPath('/workspace/data/raw/companies_house/filing_history/filinghistory_FINANCE ADVICE CENTRE LTD.json')

In [32]:
from src.data_download.companies_house import build_download_manifest
company_names = ["FINANCE ADVICE CENTRE LTD"]
FilingList = build_download_manifest(company_names)

DEBUG: {"application/pdf": {"content_length": 99014}, "application/xhtml+xml": {"content_length": 75606}}
DEBUG: {"application/pdf": {"content_length": 76337}, "application/xhtml+xml": {"content_length": 59463}}
DEBUG: {"application/pdf": {"content_length": 77083}, "application/xhtml+xml": {"content_length": 447228}}
DEBUG: {"application/pdf": {"content_length": 76428}, "application/xhtml+xml": {"content_length": 318323}}
DEBUG: {"application/pdf": {"content_length": 76231}, "application/xhtml+xml": {"content_length": 316913}}
DEBUG: {"application/pdf": {"content_length": 77819}, "application/xhtml+xml": {"content_length": 322932}}
DEBUG: {"application/pdf": {"content_length": 80057}, "application/xhtml+xml": {"content_length": 319911}}
DEBUG: {"application/pdf": {"content_length": 66737}, "application/xhtml+xml": {"content_length": 41557}}
DEBUG: {"application/pdf": {"content_length": 63262}, "application/xhtml+xml": {"content_length": 37724}}


In [34]:
from src.data_download.companies_house import download_ixbrl_filing
for filing in FilingList:
    download_ixbrl_filing(filing)

2025-08-26, application/xhtml+xml, /workspace/data/raw/companies_house/FINANCE ADVICE CENTRE LTD/MzQ3ODU4MjE4N2FkaXF6a2N4.ixbrl
2024-07-01, application/xhtml+xml, /workspace/data/raw/companies_house/FINANCE ADVICE CENTRE LTD/MzQyNzE3ODU4MGFkaXF6a2N4.ixbrl
2023-06-27, application/xhtml+xml, /workspace/data/raw/companies_house/FINANCE ADVICE CENTRE LTD/MzM4NDI4MDYwOGFkaXF6a2N4.ixbrl
2022-06-17, application/xhtml+xml, /workspace/data/raw/companies_house/FINANCE ADVICE CENTRE LTD/MzM0Mjg1MTY1MGFkaXF6a2N4.ixbrl
2021-04-28, application/xhtml+xml, /workspace/data/raw/companies_house/FINANCE ADVICE CENTRE LTD/MzI5OTI3NjMxNGFkaXF6a2N4.ixbrl
2020-10-27, application/xhtml+xml, /workspace/data/raw/companies_house/FINANCE ADVICE CENTRE LTD/MzI4MTcyOTA3NWFkaXF6a2N4.ixbrl
2019-12-23, application/xhtml+xml, /workspace/data/raw/companies_house/FINANCE ADVICE CENTRE LTD/MzI1MjkyNzY4N2FkaXF6a2N4.ixbrl
2018-12-20, application/xhtml+xml, /workspace/data/raw/companies_house/FINANCE ADVICE CENTRE LTD/MzIyMjY

In [23]:
from src.parsing.ixbrl_loader import load_ixbrl_dataframes

ixbrl_folder = projectRoot / "data" / "raw" / "companies_house"
folder_name = "FINANCE ADVICE CENTRE LTD"
facts_with_context_dict = load_ixbrl_dataframes(ixbrl_folder, folder_name)
facts_with_context_dict.keys()

Loading iXBRL files from folder: /workspace/data/raw/companies_house/FINANCE ADVICE CENTRE LTD
Processing iXBRL file: MzE4NTE0NDIyM2FkaXF6a2N4.ixbrl
Processing iXBRL file: MzI1MjkyNzY4N2FkaXF6a2N4.ixbrl
Processing iXBRL file: MzI4MTcyOTA3NWFkaXF6a2N4.ixbrl
Processing iXBRL file: MzI5OTI3NjMxNGFkaXF6a2N4.ixbrl
Processing iXBRL file: MzIyMjY3NDUxN2FkaXF6a2N4.ixbrl
Processing iXBRL file: MzM0Mjg1MTY1MGFkaXF6a2N4.ixbrl
Processing iXBRL file: MzM4NDI4MDYwOGFkaXF6a2N4.ixbrl
Processing iXBRL file: MzQ3ODU4MjE4N2FkaXF6a2N4.ixbrl
Processing iXBRL file: MzQyNzE3ODU4MGFkaXF6a2N4.ixbrl


dict_keys(['MzE4NTE0NDIyM2FkaXF6a2N4', 'MzI1MjkyNzY4N2FkaXF6a2N4', 'MzI4MTcyOTA3NWFkaXF6a2N4', 'MzI5OTI3NjMxNGFkaXF6a2N4', 'MzIyMjY3NDUxN2FkaXF6a2N4', 'MzM0Mjg1MTY1MGFkaXF6a2N4', 'MzM4NDI4MDYwOGFkaXF6a2N4', 'MzQ3ODU4MjE4N2FkaXF6a2N4', 'MzQyNzE3ODU4MGFkaXF6a2N4'])

In [ ]:
# Store to CSV files for inspection
import os

processed_dir = projectRoot / "data" / "processed" / "debug"
os.makedirs(processed_dir, exist_ok=True)

for filing_id, df in facts_with_context_dict.items():
    csv_path = processed_dir / f"{filing_id}.csv"
    df.to_csv(csv_path, index=False)

In [35]:
from src.parsing.facts_extractor import (extract_canonical_facts_dataframes,
                                         save_canonical_facts_to_csv)

extractor = extract_canonical_facts_dataframes(facts_with_context_dict)
print(extractor.keys())

processed_dir = projectRoot / "data" / "processed" / "canonical_facts"
save_canonical_facts_to_csv(extractor, processed_dir)

dict_keys(['MzE4NTE0NDIyM2FkaXF6a2N4', 'MzI1MjkyNzY4N2FkaXF6a2N4', 'MzI4MTcyOTA3NWFkaXF6a2N4', 'MzI5OTI3NjMxNGFkaXF6a2N4', 'MzIyMjY3NDUxN2FkaXF6a2N4', 'MzM0Mjg1MTY1MGFkaXF6a2N4', 'MzM4NDI4MDYwOGFkaXF6a2N4', 'MzQ3ODU4MjE4N2FkaXF6a2N4', 'MzQyNzE3ODU4MGFkaXF6a2N4'])


In [38]:
# get second key without building a list
if not extractor:
    raise ValueError("extractor is empty")
it = iter(extractor)
try:
    next(it)  # skip first
    second_key = next(it)
except StopIteration:
    raise ValueError("extractor has less than 2 items")

ff = extractor[second_key]["financial_facts"]
narrPol = extractor[second_key]["narrative_policies"]
enCompliance = extractor[second_key]["entity_compliance"]

# sense check entries
ff[ff['name'] == "frs-core:IncreaseFromDepreciationChargeForYearPropertyPlantEquipment"]
ff[ff['name'].str.contains("DepreciationCharge", case=False, na=False)]

,name,contextRef,unitRef,decimals,value,measure,entity_identifier,entity_scheme,start_date,end_date,instant,scenario,domain
34,frs-core:IncreaseFromDepreciationChargeForYear...,CURRENT_FY_PERIOD_FixturesFittings,GBP,0,97,iso4217:GBP,08948140,http://www.companieshouse.gov.uk/,2018-04-01,2019-03-31,None,None,frs-core
35,frs-core:IncreaseFromDepreciationChargeForYear...,CURRENT_FY_PERIOD_ComputerEquipment,GBP,0,"2,575",iso4217:GBP,08948140,http://www.companieshouse.gov.uk/,2018-04-01,2019-03-31,None,None,frs-core
36,frs-core:IncreaseFromDepreciationChargeForYear...,CURRENT_FY_PERIOD,GBP,0,"2,672",iso4217:GBP,08948140,http://www.companieshouse.gov.uk/,2018-04-01,2019-03-31,None,None,frs-core
